# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KMSiam/Flyrank-Internship-Week1-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Selected Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**  
**Task Type:** **Ranking / Priority Scoring** (supported by calibrated probabilistic scoring)

### Why Ranking / Scoring (and Not Just Classification)?
1. **The Decision Being Improved:**  
   In enterprise content libraries containing thousands of published URLs per client domain, editorial and SEO teams face strict operational bandwidth limits: an editorial team can realistically review and update **20 to 50 URLs per week**. The operational decision is: *"Which specific pages should our editorial team review and rewrite in this week's sprint to recover the maximum search visibility and organic revenue?"*
2. **Who Acts on the Output, and What They Do:**  
   Content strategists and SEO editors act on the output queue. For each prioritized URL, editors execute targeted actions based on transparent reason codes:
   - `refresh`: Update outdated facts, refresh statistics/citations, and deepen comprehensive section coverage.
   - `title_ctr_rewrite`: Rewrite meta titles and descriptions when search impressions are high but click-through rate lags behind expected position benchmarks.
   - `prune_or_consolidate`: Consolidate overlapping thin content competing for identical search intents.
   - `monitor`: Leave healthy, stable, or evergreen pages untouched to preserve editorial hours.
3. **What a Wrong Call Costs:**  
   - **False Positive (recommending a healthy or evergreen page for rewrite):** Wastes 2 to 5 hours of specialized editorial labor ($150–$300 in direct labor cost per article) on content that did not need intervention, while introducing unnecessary regression risk (editing high-performing content can disrupt existing keyword stability).
   - **False Negative (missing a high-traffic decaying page):** High-impression keywords silently slip from Page 1 (positions 1–10) down to Page 2–3 (positions 11–30), resulting in compounding organic traffic loss that takes months to recover.
4. **Why Binary Classification Fails Operationally (The Queue Paralysis Problem):**  
   In our starter dataset of 30,000 pages, **16,262 pages (54.2%)** exhibit active performance decay. If we treat this as a standard binary classification problem ("will this page decline: Yes/No?"), the model flags an unranked pile of over 16,000 pages. Handing 16,000 URLs to an editorial team with capacity for 50 URLs per week paralyzes human workflow. The problem is fundamentally **"Which ones first?"** We need a **Learning-to-Rank / Priority Scoring** model that orders candidate URLs by expected recovery leverage so that the top 20–50 items delivered to editors possess the highest density of true, high-impact refresh opportunities.

### The One-Paragraph Frame (following `framing-ml-problems`):
> *For enterprise editorial and SEO teams deciding which published content URLs to review and update in weekly review sprints, we will build a ranked priority scoring model from trailing 90-day search console and analytics telemetry across 32 client domains, predicting an observed performance decay opportunity score measured by Precision@20 and Precision@50 on held-out client domains. A wrong call costs wasted editorial hours ($150–$300/article) on healthy pages or compounding organic traffic loss from missed decaying search assets. A plain rule isn't enough because content decay is non-linear and multidimensional across exposure, striking position tiers, and engagement. We will claim only decision-support prioritization results on observed historical performance.*


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Resolve path reliably across Colab, repo root, and work/notebooks
RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")
if not RAW_PATH.exists():
    RAW_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not RAW_PATH.exists():
    RAW_PATH = Path("E:/FlyRank/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(RAW_PATH)

total_pages = len(df)
declining_pages = (df["trend_direction"] == "down").sum()
base_rate = declining_pages / total_pages
weekly_editor_capacity = 50

print("=== 1. LANE 2 OPERATIONAL TRIAGE & TASK TYPE ===")
print(f"Total candidate content library : {total_pages:,} URLs")
print(f"Total declining URLs (base rate) : {declining_pages:,} ({base_rate:.2%})")
print(f"Weekly editorial review capacity : {weekly_editor_capacity} URLs/week")
print(f"Time required to review all unranked declines : {declining_pages / weekly_editor_capacity:.1f} weeks (~{declining_pages / weekly_editor_capacity / 52:.1f} years!)")
print("\nOperational Conclusion: Binary classification output (16,262 positive flags) causes queue paralysis.")
print("The task must be RANKING / PRIORITY SCORING to optimize the top K (K=20, K=50) editorial review queue.")


=== 1. LANE 2 OPERATIONAL TRIAGE & TASK TYPE ===
Total candidate content library : 30,000 URLs
Total declining URLs (base rate) : 16,262 (54.21%)
Weekly editorial review capacity : 50 URLs/week
Time required to review all unranked declines : 325.2 weeks (~6.3 years!)

Operational Conclusion: Binary classification output (16,262 positive flags) causes queue paralysis.
The task must be RANKING / PRIORITY SCORING to optimize the top K (K=20, K=50) editorial review queue.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### 1. What We Predict (Target & Proxy Definition)
* **Operational Target (The Concept):** We predict **Content Refresh Opportunity** — the likelihood that a published URL with existing search visibility is experiencing active performance decay and will benefit from editorial intervention.
* **Empirical Proxy in Data:** In our starter dataset, the ground-truth target is `is_declining_label`, an observed binary outcome where `1` indicates that the page suffered significant search traffic decay ($\ge 20\%$ decline in search impressions) over trailing observation windows.
* **In the Full Warehouse Panel:** In multi-month time series, the target is observed future decline: whether a page's impressions/clicks in the subsequent 30-day window ($T_{30..60}$) decline by $\ge 15\%$ relative to its baseline window ($T_{0..30}$).

### 2. Where the Label Comes From: Observed Outcome vs. Defined Rule
Following the core principle from `skills/framing-ml-problems/SKILL.md`:
> *"The target must be observed, not defined. A label that comes from someone's rule means your model learns the rule, not the world."*

* **The Label is OBSERVED:** The label originates directly from empirical Google Search Console (GSC) and Google Analytics (GA4) telemetry comparing `impressions_last_30d` against `impressions_prev_30d`. It reflects real search user behavior and search engine indexing outcomes, not subjective human tags, internal product scores, or arbitrary rule thresholds.

### 3. The Critical Leakage Boundary (Guarding Against Target Leakage)
As documented in `docs/data-dictionary.md` and `skills/flyrank/flyrank-data/SKILL.md`:
* In this dataset, `trend_direction` is directly computed from `trend_pct` = $((	ext{impressions\_last\_30d} - 	ext{impressions\_prev\_30d}) / 	ext{impressions\_prev\_30d}) 	imes 100$.
* Specifically:
  - `down`: `trend_pct <= -20.0%`
  - `stable`: `-20.0% < trend_pct < +20.0%`
  - `up`: `trend_pct >= +20.0%`
* **THE LEAKAGE TRAP:** Because `is_declining_label = (trend_direction == 'down')`, both `trend_direction` and `trend_pct` contain the target outcome in disguise. Feeding `trend_pct` into a model produces a 100% accurate tree that learns trivial arithmetic rather than predictive patterns.
* **Safe Feature Boundary:** We strictly **quarantine and exclude** `trend_direction`, `trend_pct`, `impressions_last_30d`, and `impressions_prev_30d` from the feature matrix. All candidate features must come strictly from trailing baseline signals (`impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `content_age_days`, `days_since_last_update`, `engagement_rate`, `scroll_rate`, `word_count`, `search_volume`).


In [2]:
# Define the observed target label
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Inspect empirical label distribution
label_counts = df["is_declining_label"].value_counts().rename({0: "Stable / Growing (0)", 1: "Declining (1)"})

print("=== 2. TARGET COLUMN & LEAKAGE VERIFICATION ===")
print("Observed Label Distribution:")
print(label_counts.to_string())
print(f"\nPositive class prevalence (base rate): {df['is_declining_label'].mean():.2%}")

# Empirically prove the leakage boundary
print("\n--- LEAKAGE PROOF: trend_pct directly defines the label ---")
non_null_trend = df.dropna(subset=["trend_pct"])
matches = (non_null_trend["trend_pct"] <= -20.0) == (non_null_trend["is_declining_label"] == 1)
print(f"Fraction where (trend_pct <= -20.0) == (label == 1): {matches.mean():.4%}")
print("Min trend_pct for 'down'   :", df.loc[df['trend_direction'] == 'down', 'trend_pct'].min())
print("Max trend_pct for 'down'   :", df.loc[df['trend_direction'] == 'down', 'trend_pct'].max())
print("Min trend_pct for 'stable' :", df.loc[df['trend_direction'] == 'stable', 'trend_pct'].min())

print("\n[SAFETY ACTION] Columns strictly QUARANTINED (never features):")
print("  - trend_direction (defines target)")
print("  - trend_pct       (leakage: mathematical source of target)")
print("  - impressions_last_30d / impressions_prev_30d (sub-window components of target)")


=== 2. TARGET COLUMN & LEAKAGE VERIFICATION ===
Observed Label Distribution:
is_declining_label
Declining (1)           16262
Stable / Growing (0)    13738

Positive class prevalence (base rate): 54.21%

--- LEAKAGE PROOF: trend_pct directly defines the label ---
Fraction where (trend_pct <= -20.0) == (label == 1): 99.8084%
Min trend_pct for 'down'   : -100.0
Max trend_pct for 'down'   : -20.0
Min trend_pct for 'stable' : -20.0

[SAFETY ACTION] Columns strictly QUARANTINED (never features):
  - trend_direction (defines target)
  - trend_pct       (leakage: mathematical source of target)
  - impressions_last_30d / impressions_prev_30d (sub-window components of target)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### 1. The Metric: Precision@K (Specifically Precision@20 and Precision@50)
* **Definition:** Out of the top $K$ pages flagged by our ranking model, what proportion are genuine decaying content opportunities?
  $$\text{Precision@K} = \frac{1}{K} \sum_{i=1}^{K} y_{(i)}$$
  where $y_{(i)} \in \{0, 1\}$ is the true observed decline indicator of the $i$-th ranked page.

### 2. Why Precision@K is the Only Defensible Primary Metric:
1. **Direct Alignment with Human Capacity:** Editorial workflows operate in discrete weekly sprints ($K = 20$ for an individual editor, $K = 50$ for an editorial squad). An algorithm is evaluated exclusively on the quality of the top slots it places in front of human eyes.
2. **Asymmetric Error Cost:** A false positive in the top 20 directly costs 2–5 hours of high-value human labor investigating a healthy article. A ranking model with high overall ROC-AUC that fails at the top of the queue is operationally useless.
3. **Recall is Irrelevant:** Because 16,262 pages are declining, 100% recall is unattainable and undesirable — no team can review 16,000 pages. Precision in the top $K$ slots is what delivers business ROI.
4. **Secondary Guardrails:** We report **Client-Holdout ROC-AUC** as a diagnostic check for global ranking consistency across unseen domains, and **Lift over Base Rate** ($P@K / \text{Base Rate}$).

### 3. What Number Means "Good"?
* **Random Guessing Benchmark:** **54.21%** (equal to the empirical base rate of decline). A random draw delivers ~11 true opportunities in the top 20.
* **Domain Heuristic Baseline (Stale & Visible Rule):** **~45.0%** at Top 20 (simple rules perform poorly at the very top because top-impression URLs are often evergreen pillars).
* **Target "Good" Model Threshold:**
  - **Precision@20 $\ge 75.0\%$** ($\ge 15$ out of top 20 are true opportunities; $\ge 1.38\times$ lift over base rate).
  - **Precision@50 $\ge 70.0\%$** ($\ge 35$ out of top 50 are true opportunities; $\ge 1.29\times$ lift over base rate).
  - Evaluated on **held-out client domains** (GroupKFold by `client_id`) to prove generalization to new clients.


In [3]:
def precision_at_k(y_true, scores, k: int = 20) -> float:
    """Calculates Precision@K for a ranking score against true binary labels."""
    frame = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    top_k = frame.sort_values("score", ascending=False).head(k)
    return float(top_k["y"].mean()) if len(top_k) > 0 else 0.0

# Calculate baseline numbers
random_expected_p20 = base_rate
random_expected_p50 = base_rate
target_good_p20 = 0.75
target_good_p50 = 0.70

metrics_defense = pd.DataFrame([
    {
        "Method / Benchmark": "Random Guessing (Base Rate)",
        "Precision@20": f"{random_expected_p20:.1%}",
        "Precision@50": f"{random_expected_p50:.1%}",
        "True Opportunities in Top-20": f"{random_expected_p20 * 20:.1f} / 20",
        "Operational Verdict": "Unacceptable (wastes ~9 editor hours per batch)"
    },
    {
        "Method / Benchmark": "Heuristic Rule (Stale x Visible)",
        "Precision@20": "45.0%",
        "Precision@50": "44.0%",
        "True Opportunities in Top-20": "9.0 / 20",
        "Operational Verdict": "Fragile (fails at top of queue due to evergreen bias)"
    },
    {
        "Method / Benchmark": "Target 'Good' ML Ranking Model",
        "Precision@20": f">= {target_good_p20:.1%}",
        "Precision@50": f">= {target_good_p50:.1%}",
        "True Opportunities in Top-20": f">= {target_good_p20 * 20:.0f} / 20",
        "Operational Verdict": "High-leverage decision support (delivers strong editor ROI)"
    }
])

print("=== 3. SUCCESS METRIC DEFENSE & BENCHMARK THRESHOLDS ===")
print(metrics_defense.to_string(index=False))


=== 3. SUCCESS METRIC DEFENSE & BENCHMARK THRESHOLDS ===
              Method / Benchmark Precision@20 Precision@50 True Opportunities in Top-20                                         Operational Verdict
     Random Guessing (Base Rate)        54.2%        54.2%                    10.8 / 20             Unacceptable (wastes ~9 editor hours per batch)
Heuristic Rule (Stale x Visible)        45.0%        44.0%                     9.0 / 20       Fragile (fails at top of queue due to evergreen bias)
  Target 'Good' ML Ranking Model     >= 75.0%     >= 70.0%                   >= 15 / 20 High-leverage decision support (delivers strong editor ROI)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### 1. The Unit of Analysis (Explicit Definition)
**One row = One published content item (`content_id`) within a client domain (`client_id`), evaluated over a trailing 90-day search console and analytics telemetry window.**

### 2. Dataset Properties & Gotchas (from `skills/flyrank/flyrank-data/SKILL.md`):
- **Grain Verification:** Exactly 30,000 rows, and exactly 30,000 unique `content_id` values (`nunique == len(df)`). Every row is a distinct content page.
- **Client Grouping:** Covers 32 pseudonymized client domains. Client IDs are used strictly for **grouped train/test splits** (never as predictive features) to prevent data leakage between domains.
- **Rate Scale Gotcha:** Rates are $0–100$ percentages (`ctr = 0.76` means $0.76\%$, not $76\%$). This applies to `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and `trend_pct`.
- **Zero Position Gotcha:** `avg_position = 0` indicates **no rank data** (1,205 rows), not rank zero. Must be treated as missing/unranked.
- **Systematic Missingness:** Missingness follows `content_type` lines (e.g. `feedly article` has no keyword search volume/competition; 7,699 rows lack `word_count`). Blind `fillna(0)` injects categorical bias; missingness flags must be used.


In [4]:
# Define candidate feature view for Lane 2 (Content Refresh Opportunity)
candidate_cols = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "engagement_rate",
    "scroll_rate",
    "is_declining_label"
]

df_slice = df[candidate_cols].copy()

print("=== 4. UNIT OF ANALYSIS VERIFICATION ===")
print(f"DataFrame Shape            : {df_slice.shape[0]:,} rows x {df_slice.shape[1]} columns")
print(f"Unique content_id (grain)  : {df_slice['content_id'].nunique():,} (Unique Grain Confirmed: {df_slice['content_id'].nunique() == len(df_slice)})")
print(f"Unique client_id (domains) : {df_slice['client_id'].nunique()} distinct client accounts")
print(f"avg_position == 0 rows     : {(df_slice['avg_position'] == 0).sum():,} (represents unranked / no search data)")
print(f"Missing scroll_rate values : {df_slice['scroll_rate'].isna().sum():,}")

print("\n--- Real Dataframe Slice: One row = One published content item (Top 5 rows) ---")
display_cols = ["content_id", "client_id", "content_type", "impressions_90d", "ctr", "avg_position", "days_since_last_update", "is_declining_label"]
print(df_slice[display_cols].head(5).to_string(index=False))


=== 4. UNIT OF ANALYSIS VERIFICATION ===
DataFrame Shape            : 30,000 rows x 12 columns
Unique content_id (grain)  : 30,000 (Unique Grain Confirmed: True)
Unique client_id (domains) : 32 distinct client accounts
avg_position == 0 rows     : 1,205 (represents unranked / no search data)
Missing scroll_rate values : 125

--- Real Dataframe Slice: One row = One published content item (Top 5 rows) ---
          content_id         client_id    content_type  impressions_90d  ctr  avg_position  days_since_last_update  is_declining_label
content_304f48230142 client_f369cb89fc keyword article             3803 0.76          10.6                      20                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320 0.05          20.3                      25                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581 0.09          36.5                      20                   1
content_331d6c4de07b client_19581e27de keyword articl

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Four Concrete Reasons Why a Fixed Rule Fails:
1. **The Evergreen Paradox (Failure at the Top of the Queue):**  
   A sensible rule like `days_since_last_update >= 90 AND impressions_90d >= 500` matches 6,575 pages. But when sorted by raw search impressions (the intuitive way humans rank), its top 20 picks achieve only **45.0% precision** — worse than flipping a coin! Why? Because high-authority pillar articles and homepages naturally generate immense search impressions and remain stably indexed for months without updates. A blunt rule mistakes evergreen authority for urgent refresh need.
2. **Non-Linear Striking Position Dynamics:**  
   Google's organic click curve is brutally non-linear: losing 2 positions from position 4 to 6 causes a $\sim 50\%$ collapse in organic clicks, whereas losing 2 positions from position 28 to 30 has virtually zero real-world traffic impact. A fixed boolean threshold cannot weigh position tiers adaptively against impression volume.
3. **Multidimensional Signal Entanglement:**  
   A high-exposure page with low CTR points to a title/snippet relevance issue. A high-exposure page with high bounce rate points to outdated content body depth. A page with declining impressions but high commercial intent keyword CPC represents high financial urgency. No hand-crafted nested if-statement can balance exposure, position, CTR deficits, keyword value, and engagement simultaneously without collapsing into unmaintainable heuristic spaghetti.
4. **Continuous Priority Calibration vs. Blunt Boolean Bins:**  
   A fixed rule outputs an undifferentiated binary bucket of 6,575 pages. An ML model learns the non-linear interaction weights across all baseline signals, outputting a continuous, calibrated opportunity score that creates a high-precision, finely ordered ranking queue.


In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold

# 1. EVALUATE THE FIXED RULE BASELINE (Stale x Visible)
# Rule: Not updated in >= 90 days, meaningful search exposure (impressions >= 500)
rule_mask = (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 500)
df["rule_score"] = np.where(rule_mask, df["impressions_90d"], 0.0)

rule_p20 = precision_at_k(df["is_declining_label"], df["rule_score"], k=20)
rule_p50 = precision_at_k(df["is_declining_label"], df["rule_score"], k=50)

# 2. BUILD AN INTERPRETABLE ML MODEL ON LEGITIMATE FEATURES (NO LEAKAGE!)
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "engagement_rate",
    "scroll_rate"
]

X = df[feature_cols].copy()
# Clean avg_position gotcha (0 means unranked)
X["avg_position"] = X["avg_position"].replace(0, np.nan)
X = X.fillna(X.median())
y = df["is_declining_label"]
groups = df["client_id"]

# 5-fold Client-Holdout Cross-Validation (Strictly unseen clients in validation!)
gkf = GroupKFold(n_splits=5)
oof_scores = np.zeros(len(df))

for train_idx, val_idx in gkf.split(X, y, groups):
    clf = DecisionTreeClassifier(max_depth=3, random_state=42)
    clf.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_scores[val_idx] = clf.predict_proba(X.iloc[val_idx])[:, 1]

# Add tiny secondary tie-breaker on impression exposure
df["ml_opportunity_score"] = oof_scores + (df["impressions_90d"] / df["impressions_90d"].max()) * 1e-4

ml_p20 = precision_at_k(df["is_declining_label"], df["ml_opportunity_score"], k=20)
ml_p50 = precision_at_k(df["is_declining_label"], df["ml_opportunity_score"], k=50)

# Summary Comparison
comparison = pd.DataFrame([
    {
        "Evaluation Method": "Random Baseline",
        "Precision@20": f"{base_rate:.1%}",
        "Precision@50": f"{base_rate:.1%}",
        "Lift over Base": "1.00x",
        "Operational Result": "No prioritization (flips a coin)"
    },
    {
        "Evaluation Method": "Fixed Rule (Stale x Visible)",
        "Precision@20": f"{rule_p20:.1%}",
        "Precision@50": f"{rule_p50:.1%}",
        "Lift over Base": f"{rule_p20 / base_rate:.2f}x",
        "Operational Result": "Fails at top of queue (high evergreen false alarms)"
    },
    {
        "Evaluation Method": "Client-Holdout ML Ranking (OOF)",
        "Precision@20": f"{ml_p20:.1%}",
        "Precision@50": f"{ml_p50:.1%}",
        "Lift over Base": f"{ml_p20 / base_rate:.2f}x",
        "Operational Result": "High-density actionable queue across held-out clients"
    }
])

print("=== 5. EMPIRICAL PROOF: WHY ML BEATS A FIXED RULE ===")
print(comparison.to_string(index=False))

# Show top 5 ML ranked pages
top5_ml = df.sort_values("ml_opportunity_score", ascending=False).head(5)
print("\n--- Top 5 Pages Selected by ML Priority Scoring ---")
print(top5_ml[["content_id", "client_id", "impressions_90d", "avg_position", "days_since_last_update", "is_declining_label"]].to_string(index=False))


=== 5. EMPIRICAL PROOF: WHY ML BEATS A FIXED RULE ===
              Evaluation Method Precision@20 Precision@50 Lift over Base                                    Operational Result
                Random Baseline        54.2%        54.2%          1.00x                      No prioritization (flips a coin)
   Fixed Rule (Stale x Visible)        45.0%        44.0%          0.83x   Fails at top of queue (high evergreen false alarms)
Client-Holdout ML Ranking (OOF)        70.0%        78.0%          1.29x High-density actionable queue across held-out clients

--- Top 5 Pages Selected by ML Priority Scoring ---
          content_id         client_id  impressions_90d  avg_position  days_since_last_update  is_declining_label
content_21b3d827d451 client_3fdba35f04            38894          39.4                      14                   1
content_8330b826f55b client_3fdba35f04            24808          16.5                      26                   1
content_0636f7bcfa05 client_3fdba35f04     

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
